# 🔍 Face Finder — Bulk Photo Search
Find a specific person across a large collection of photos using face recognition.

**Library used:** [`face_recognition`](https://github.com/agner/face_recognition) (built on dlib) — simple, accurate, easy to install.

---
## Step 0 — Install Dependencies
Run this cell once. Restart the kernel after installation if prompted.

In [18]:
# Install required libraries (run once)
import sys
!{sys.executable} -m pip install face_recognition Pillow tqdm ipywidgets --quiet
print("✅ All dependencies installed!")

## Step 1 — Import Libraries

In [19]:
import face_recognition
import numpy as np
from PIL import Image, ImageDraw
import os
import shutil
import zipfile
from pathlib import Path
from tqdm.notebook import tqdm
from IPython.display import display, HTML, Image as IPImage
import ipywidgets as widgets
import io
import json

print("✅ Libraries loaded successfully!")

## Step 2 — Upload Your Bulk Photo Collection

Upload all the photos you want to search through (group photos, event photos, etc.).
Supported formats: **JPG, JPEG, PNG, BMP, WEBP**

In [20]:
# ── Configuration ──────────────────────────────────────────────
BULK_FOLDER   = Path("bulk_photos")       # folder for the bulk collection
RESULTS_FOLDER = Path("matched_photos")   # folder where matches are saved
SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
TOLERANCE     = 0.55   # lower = stricter match (0.4–0.6 recommended)

BULK_FOLDER.mkdir(exist_ok=True)
RESULTS_FOLDER.mkdir(exist_ok=True)

print(f"📁 Bulk folder  : {BULK_FOLDER.resolve()}")
print(f"📁 Results folder: {RESULTS_FOLDER.resolve()}")
print(f"⚙️  Tolerance     : {TOLERANCE}  (lower = stricter)")

In [21]:
# Upload bulk photos via widget
bulk_uploader = widgets.FileUpload(
    accept="image/*",
    multiple=True,
    description="Upload bulk photos",
    layout=widgets.Layout(width="400px")
)

save_btn = widgets.Button(
    description="Save to bulk_photos/",
    button_style="primary",
    icon="upload"
)
out = widgets.Output()

def save_bulk_photos(b):
    out.clear_output()
    with out:
        if not bulk_uploader.value:
            print("⚠️  No files selected.")
            return
        count = 0
        val = bulk_uploader.value
        
        # --- Handle different ipywidgets versions ---
        if isinstance(val, dict):
            # Old style: {filename: {content: bytes, ...}}
            for name, file_info in val.items():
                save_path = BULK_FOLDER / name
                save_path.write_bytes(file_info["content"])
                count += 1
        elif isinstance(val, (list, tuple)):
            # New style: list/tuple of dicts with 'name' and 'content'
            for item in val:
                if isinstance(item, dict) and 'name' in item and 'content' in item:
                    save_path = BULK_FOLDER / item['name']
                    save_path.write_bytes(item['content'])
                    count += 1
                elif isinstance(item, (list, tuple)) and len(item) >= 2:
                    # Some versions return (name, content, ...)
                    save_path = BULK_FOLDER / str(item[0])
                    save_path.write_bytes(item[1])
                    count += 1
                else:
                    print(f"⚠️  Unexpected file entry format: {type(item)}")
        else:
            print(f"⚠️  Unknown uploader value type: {type(val)}")
            return
        
        print(f"✅ Saved {count} photo(s) to '{BULK_FOLDER}/'")

save_btn.on_click(save_bulk_photos)
display(widgets.VBox([bulk_uploader, save_btn, out]))

> **Alternative:** If you already have photos on disk, just copy them into the `bulk_photos/` folder manually and skip the uploader above.

## Step 3 — Analyze Faces in the Bulk Collection

This step scans all photos and encodes every face found. It may take a few minutes for large collections.

In [22]:
def load_image_safe(path):
    """Load image and convert to RGB (handles PNG transparency, EXIF rotation)."""
    try:
        img = Image.open(path)
        img = img.convert("RGB")
        return np.array(img)
    except Exception as e:
        print(f"  ⚠️  Could not load {path.name}: {e}")
        return None


def analyze_bulk_photos(folder):
    """Encode all faces found in the bulk folder."""
    photo_paths = [p for p in folder.iterdir() if p.suffix.lower() in SUPPORTED_EXTS]

    if not photo_paths:
        print("❌ No photos found in bulk_photos/. Upload photos in Step 2 first.")
        return {}

    print(f"🔍 Scanning {len(photo_paths)} photos...")
    database = {}  # path → list of face encodings
    face_counts = {"total_photos": 0, "with_faces": 0, "total_faces": 0}

    for photo_path in tqdm(photo_paths, desc="Analyzing"):
        img_array = load_image_safe(photo_path)
        if img_array is None:
            continue

        face_counts["total_photos"] += 1
        locations = face_recognition.face_locations(img_array, model="hog")
        if not locations:
            continue

        encodings = face_recognition.face_encodings(img_array, locations)
        if encodings:
            database[str(photo_path)] = encodings
            face_counts["with_faces"] += 1
            face_counts["total_faces"] += len(encodings)

    print(f"\n📊 Analysis complete!")
    print(f"   Photos scanned : {face_counts['total_photos']}")
    print(f"   Photos with faces: {face_counts['with_faces']}")
    print(f"   Total faces found: {face_counts['total_faces']}")
    return database


# Run the analysis
photo_database = analyze_bulk_photos(BULK_FOLDER)
print("\n✅ Database ready! Proceed to Step 4.")

## Step 4 — Upload Your Reference Photo (Selfie)

Upload a clear photo of the person you want to find. A well-lit, front-facing photo works best.

In [23]:
selfie_uploader = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload selfie/reference",
    layout=widgets.Layout(width="400px")
)

selfie_out = widgets.Output()
reference_encoding = [None]  # store encoding in a list so closure can modify it

# Step 4 — Upload Reference Photo
selfie_uploader = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload selfie/reference",
    layout=widgets.Layout(width="400px")
)

selfie_out = widgets.Output()
reference_encoding = [None]  # mutable container

def on_selfie_upload(change):
    with selfie_out:
        selfie_out.clear_output()
        val = selfie_uploader.value
        if not val:
            return

        # Extract file name and bytes
        name = None
        img_bytes = None
        if isinstance(val, dict):
            name, info = next(iter(val.items()))
            img_bytes = info.get("content") if isinstance(info, dict) else info
        elif isinstance(val, (list, tuple)) and len(val) > 0:
            item = val[0]
            if isinstance(item, dict):
                name = item.get("name")
                img_bytes = item.get("content")
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                name = str(item[0])
                img_bytes = item[1]

        if not name or not img_bytes:
            print("❌ Could not read uploaded file. Raw value type:", type(val))
            return

        # Preview
        from IPython.display import Image as IPImage
        display(IPImage(data=img_bytes, width=200))
        print(f"📷 Reference photo: {name}")

        # Face encoding
        from PIL import Image
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        img_array = np.array(img)
        locations = face_recognition.face_locations(img_array, model="hog")
        if not locations:
            print("❌ No face detected. Try a clearer, front‑facing photo.")
            reference_encoding[0] = None
            return

        if len(locations) > 1:
            print(f"⚠️ {len(locations)} faces detected – using largest.")
            areas = [(b - t) * (r - l) for (t, r, b, l) in locations]
            locations = [locations[areas.index(max(areas))]]

        encodings = face_recognition.face_encodings(img_array, locations)
        if encodings:
            reference_encoding[0] = encodings[0]
            print("✅ Face encoded. Proceed to Step 5.")
        else:
            print("❌ Face encoding failed. Try another photo.")
            reference_encoding[0] = None

# Connect the handler
selfie_uploader.observe(on_selfie_upload, names="value")

# Display
display(widgets.VBox([selfie_uploader, selfie_out]))

## Step 5 — Find Matching Photos

In [24]:
def find_matches(database, ref_encoding, tolerance=TOLERANCE):
    """Search the database for photos containing the reference face."""
    if ref_encoding is None:
        print("❌ No reference face loaded. Complete Step 4 first.")
        return []

    if not database:
        print("❌ Photo database is empty. Complete Step 3 first.")
        return []

    print(f"🔍 Searching {len(database)} photos with tolerance={tolerance}...\n")
    matches = []

    for photo_path, encodings in tqdm(database.items(), desc="Matching"):
        distances = face_recognition.face_distance(encodings, ref_encoding)
        if any(d <= tolerance for d in distances):
            best_dist = float(min(distances))
            matches.append({"path": photo_path, "distance": best_dist})

    # Sort best match first
    matches.sort(key=lambda x: x["distance"])

    print(f"\n🎯 Found {len(matches)} matching photo(s)!")
    return matches


matched_photos = find_matches(photo_database, reference_encoding[0])

## Step 6 — Preview Matching Photos

In [25]:
def draw_faces_on_image(img_array, ref_encoding, tolerance=TOLERANCE):
    """Draw bounding boxes around matched faces."""
    locations = face_recognition.face_locations(img_array, model="hog")
    encodings = face_recognition.face_encodings(img_array, locations)

    pil_img = Image.fromarray(img_array)
    draw = ImageDraw.Draw(pil_img)

    for (top, right, bottom, left), enc in zip(locations, encodings):
        dist = face_recognition.face_distance([ref_encoding], enc)[0]
        color = "#00FF88" if dist <= tolerance else "#FF4444"
        for offset in range(3):  # thick border
            draw.rectangle(
                [left - offset, top - offset, right + offset, bottom + offset],
                outline=color
            )
        label = f"{dist:.2f}"
        draw.text((left, bottom + 4), label, fill=color)

    return pil_img


def preview_matches(matches, ref_encoding, max_preview=10):
    if not matches:
        print("No matches to preview.")
        return

    display(HTML("<h3>✅ Matching Photos (green box = matched face)</h3>"))

    for i, m in enumerate(matches[:max_preview]):
        img_array = load_image_safe(Path(m["path"]))
        if img_array is None:
            continue

        annotated = draw_faces_on_image(img_array, ref_encoding)

        # Resize for display
        w, h = annotated.size
        scale = min(1.0, 500 / max(w, h))
        annotated = annotated.resize((int(w * scale), int(h * scale)))

        buf = io.BytesIO()
        annotated.save(buf, format="JPEG", quality=85)

        fname = Path(m['path']).name
        conf = (1 - m['distance']) * 100
        display(HTML(f"<b>{i+1}. {fname}</b> — confidence: {conf:.1f}%"))
        display(IPImage(data=buf.getvalue()))
        print()

    if len(matches) > max_preview:
        print(f"... and {len(matches) - max_preview} more (download to see all).")


preview_matches(matched_photos, reference_encoding[0])

## Step 7 — Download Matched Photos as ZIP

In [26]:
def copy_matches_to_results(matches, results_folder):
    """Copy matched photos to the results folder."""
    # Clear previous results
    shutil.rmtree(results_folder, ignore_errors=True)
    results_folder.mkdir()

    for m in matches:
        src = Path(m["path"])
        dst = results_folder / src.name
        shutil.copy2(src, dst)

    print(f"✅ Copied {len(matches)} photo(s) to '{results_folder}/'")


def create_zip(matches, zip_name="matched_photos.zip"):
    """Bundle all matched photos into a ZIP file."""
    if not matches:
        print("❌ No matches to package.")
        return None

    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for m in matches:
            p = Path(m["path"])
            zf.write(p, arcname=p.name)

    size_mb = Path(zip_name).stat().st_size / 1_000_000
    print(f"\n📦 ZIP created: {zip_name}  ({size_mb:.2f} MB, {len(matches)} photos)")
    return zip_name


# Copy & zip
copy_matches_to_results(matched_photos, RESULTS_FOLDER)
zip_path = create_zip(matched_photos)

In [ ]:
# Generate a one-click download link (works in JupyterLab & classic Notebook)
import base64

def download_link(file_path, label="⬇️ Download matched_photos.zip"):
    if file_path is None or not Path(file_path).exists():
        print("❌ ZIP file not found.")
        return
    data = Path(file_path).read_bytes()
    b64 = base64.b64encode(data).decode()
    href = (
        f'<a href="data:application/zip;base64,{b64}" '
        f'download="{Path(file_path).name}" '
        f'style="font-size:18px; padding:10px 20px; background:#0066ff; '
        f'color:white; border-radius:8px; text-decoration:none;">{label}</a>'
    )
    display(HTML(href))
    print(f"\nAlternatively, find the files in: {RESULTS_FOLDER.resolve()}")


download_link(zip_path)

## Step 8 — (Optional) Save a Match Report

In [ ]:
def save_report(matches, report_path="match_report.json"):
    report = [
        {
            "file": Path(m["path"]).name,
            "distance": round(m["distance"], 4),
            "confidence_pct": round((1 - m["distance"]) * 100, 1),
        }
        for m in matches
    ]
    Path(report_path).write_text(json.dumps(report, indent=2))
    print(f"📄 Report saved: {report_path}")
    display(HTML("<pre>" + json.dumps(report[:5], indent=2) + ("\n..." if len(report) > 5 else "") + "</pre>"))


save_report(matched_photos)

---
## ⚙️ Tuning Tips

| Setting | Effect |
|---|---|
| `TOLERANCE = 0.40` | Very strict — fewer false positives, may miss some |
| `TOLERANCE = 0.55` | Balanced (default) |
| `TOLERANCE = 0.65` | Lenient — catches more but may include wrong people |
| `model="cnn"` in `face_locations()` | More accurate, but much slower (needs GPU for speed) |

To adjust tolerance, change `TOLERANCE` in **Step 2** and re-run Steps 3–7.

---
*Built with [face_recognition](https://github.com/ageitgey/face_recognition) by Adam Geitgey — 99.38% accuracy on LFW benchmark.*